In [19]:
# Represent (partial) triangulations with sets of edges

# Shortcut for writing edges
def e(x,y):
    return frozenset({x,y})

# Helper function to generate external edges of an n-gon:
def ngon(n):
    return frozenset(e(i%n+1,(i+1)%n+1) for i in range(n))

# Initial partial triangulation
init_pent = ngon(5) | {e(1,4)}

# Dictionary of all currently assigned resolutions
# A resolution is an ordered pair (partial_triangulation, distinguished_edge)
resolutions = {}
# Assign resolution to initial partial triangulation
#resolutions[init_pent] = (init_pent, e(1,3))

In [20]:
init_pent

frozenset({frozenset({3, 4}),
           frozenset({1, 4}),
           frozenset({1, 5}),
           frozenset({2, 3}),
           frozenset({1, 2}),
           frozenset({4, 5})})

In [21]:
# Convert a resolution into a full triangulation
def resolve(resolution):
    return resolution[0] | {resolution[1]}

# Determine whether an edge of a polygon triangulation is external
def is_external(edge, t):
    assert(len(edge) == 2)
    n = max([v for e in t for v in e])
    e = tuple(edge)
    d = (e[0] - e[1]) % n
    return (d == 1 or d == n-1)

# retrieve only the internal edges of a (partial) triangulation
def internal_edges(t):
    return frozenset({e for e in t if not is_external(e, t)})

# pretty print for (partial) triangulations
def pp(t):
    return "{" + ", ".join(str(tuple(e)) for e in internal_edges(t)) + "}"

In [22]:
# Get quadrilateral spanned by an internal edge of a (full) triangulation
def get_quad(e, t):
    assert(not is_external(e,t))
    incident_edges = [ee for ee in t - {e} if e & ee]
    q = frozenset({v for e1 in incident_edges for e2 in incident_edges for v in e1 & e2 if e1 != e2})
    assert(len(q) == 4)
    return q

# Edge flip mutation of triangulation at a given edge
def mutate(t, e):
    q = get_quad(e, t)
    ee = q - e
    return (t - {e}) | {ee}

In [23]:
hept_example = ngon(7) | {e(1,3),e(3,5),e(3,6),e(1,6)}
pp(mutate(hept_example, e(3,6)))

'{(1, 6), (1, 3), (1, 5), (3, 5)}'

In [24]:
# Find all mutable edges adjacent (in the corresponding quiver) to a given edge in a triangulation
def adjacent_edges(e, t):
    q = get_quad(e, t)
    return [ee for ee in t - {e} if not is_external(ee, t) and ee < q]

# Generate all resolutions that differ at one mutation adjacent
# to the resolved edge
def adjacent_resolutions(r):
    e = r[1]
    t = resolve(r)
    cols = set()
    for ee in adjacent_edges(e,t):
        tt = mutate(t, ee)
        assert(len(tt-t) == 1)
        new_e = list(tt - t)[0]
        cols.add((tt - {e}, e))
    return cols

# Generate all resolutions that collide with a given resolution
def generate_collisions(r):
    visited = {r}
    remaining = adjacent_resolutions(r)
    while remaining:
        rr = remaining.pop()
        visited.add(rr)
        remaining |= adjacent_resolutions(rr) - visited
    return visited - {r}

In [25]:
for (t,ee) in generate_collisions((hept_example - {e(3,6)}, e(3,6))):
    print(pp(t))

{(1, 6), (4, 6), (2, 6)}
{(1, 6), (3, 5), (2, 6)}
{(4, 6), (1, 3), (3, 7)}
{(1, 6), (4, 6), (1, 3)}
{(1, 3), (3, 5), (3, 7)}


In [33]:
def mutate_resolution(r):
    t = resolve(r)
    e = r[1]
    tt = mutate(t, e)
    assert(len(tt-t) == 1)
    ee = list(tt - t)[0]
    assert(tt-{ee} == r[0])
    return (r[0], ee)

# Pretty print resolution
def ppr(r):
    return "({}, {})".format(pp(r[0]), set(r[1]))

def propagate_resolutions(r, rz):
    for rr in generate_collisions(r):
        mutated = mutate_resolution(rr)
        print(ppr(r) + " requires " + ppr(mutated))
        if rr[0] in rz:
            if rz[rr[0]] != mutated:
                print("Conflicting resolution for {}!".format(pp(rr[0])))
                raise Exception()
        else:
            rz[rr[0]] = mutated
            propagate_resolutions(rz[rr[0]], rz)

# Wrapper to avoid constructing initial resolution map
def find_resolutions(r):
    rz = {}
    rz[r[0]] = r
    propagate_resolutions(r, rz)
    return rz

In [30]:
find_resolutions((hept_example - {e(3,6)}, e(3,6)))

({(1, 6), (1, 3), (3, 5)}, {3, 6}) requires ({(1, 6), (4, 6), (2, 6)}, {2, 4})
({(1, 6), (4, 6), (2, 6)}, {2, 4}) requires ({(1, 6), (1, 4), (4, 6)}, {1, 3})
({(1, 6), (1, 4), (4, 6)}, {1, 3}) requires ({(1, 6), (3, 6), (4, 6)}, {2, 6})
({(1, 6), (3, 6), (4, 6)}, {2, 6}) requires ({(2, 4), (2, 7), (4, 6)}, {4, 7})
({(2, 4), (2, 7), (4, 6)}, {4, 7}) requires ({(2, 7), (5, 7), (3, 7)}, {3, 5})
({(2, 7), (5, 7), (3, 7)}, {3, 5}) requires ({(3, 6), (2, 7), (3, 7)}, {4, 6})
({(3, 6), (2, 7), (3, 7)}, {4, 6}) requires ({(2, 7), (3, 7), (4, 7)}, {5, 7})
({(2, 7), (3, 7), (4, 7)}, {5, 7}) requires ({(2, 7), (2, 5), (3, 5)}, {2, 6})
({(2, 7), (2, 5), (3, 5)}, {2, 6}) requires ({(1, 6), (2, 5), (3, 5)}, {1, 5})
({(1, 6), (2, 5), (3, 5)}, {1, 5}) requires ({(1, 4), (1, 3), (5, 7)}, {4, 7})
({(1, 4), (1, 3), (5, 7)}, {4, 7}) requires ({(4, 6), (1, 3), (3, 7)}, {3, 6})
({(4, 6), (1, 3), (3, 7)}, {3, 6}) requires ({(1, 6), (1, 3), (3, 5)}, {1, 5})
Conflicting resolution for {(1, 6), (1, 3), (3, 5)}!

Exception: 

In [34]:
find_resolutions((init_pent, e(1,3)))

({(1, 4)}, {1, 3}) requires ({(3, 5)}, {2, 5})
({(3, 5)}, {2, 5}) requires ({(2, 4)}, {1, 4})
({(2, 4)}, {1, 4}) requires ({(1, 3)}, {3, 5})
({(1, 3)}, {3, 5}) requires ({(2, 5)}, {2, 4})
({(2, 5)}, {2, 4}) requires ({(1, 4)}, {1, 3})


{frozenset({frozenset({3, 4}),
            frozenset({1, 4}),
            frozenset({1, 5}),
            frozenset({2, 3}),
            frozenset({1, 2}),
            frozenset({4, 5})}): (frozenset({frozenset({3, 4}),
             frozenset({1, 4}),
             frozenset({1, 5}),
             frozenset({2, 3}),
             frozenset({1, 2}),
             frozenset({4, 5})}),
  frozenset({1, 3})),
 frozenset({frozenset({3, 4}),
            frozenset({2, 3}),
            frozenset({1, 2}),
            frozenset({4, 5}),
            frozenset({1, 5}),
            frozenset({3, 5})}): (frozenset({frozenset({3, 4}),
             frozenset({2, 3}),
             frozenset({1, 2}),
             frozenset({4, 5}),
             frozenset({1, 5}),
             frozenset({3, 5})}),
  frozenset({2, 5})),
 frozenset({frozenset({3, 4}),
            frozenset({2, 3}),
            frozenset({1, 2}),
            frozenset({4, 5}),
            frozenset({2, 4}),
            frozenset({1, 5})}): (froze

In [37]:
find_resolutions((ngon(6) | {e(1,5),e(1,3)}, e(3,5)))

({(1, 5), (1, 3)}, {3, 5}) requires ({(1, 5), (2, 5)}, {2, 4})
({(1, 5), (2, 5)}, {2, 4}) requires ({(1, 4), (1, 5)}, {1, 3})
({(1, 4), (1, 5)}, {1, 3}) requires ({(1, 5), (3, 5)}, {2, 5})
({(1, 5), (3, 5)}, {2, 5}) requires ({(2, 4), (1, 5)}, {1, 4})
({(2, 4), (1, 5)}, {1, 4}) requires ({(4, 6), (1, 3)}, {3, 6})
({(4, 6), (1, 3)}, {3, 6}) requires ({(1, 3), (3, 5)}, {1, 5})
({(1, 3), (3, 5)}, {1, 5}) requires ({(1, 4), (1, 3)}, {4, 6})
({(1, 4), (1, 3)}, {4, 6}) requires ({(3, 6), (1, 3)}, {3, 5})
({(3, 6), (1, 3)}, {3, 5}) requires ({(1, 5), (2, 5)}, {2, 4})
({(3, 6), (1, 3)}, {3, 5}) requires ({(1, 5), (1, 3)}, {1, 4})
Conflicting resolution for {(1, 5), (1, 3)}!


Exception: 